# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulsamad010/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES['fact_daily_sample']}
""").df()

,rows
0,11694072


## Answer

Unit of analysis: One row represents the daily search performance for a content record on a specific reporting date.

Time window: The available time period is determined from the `report_date` field and verified by the query below.

In [7]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily_sample']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-06-01,2026-06-30,11694072


## 2. Fields: feature / label / context / excluded

## Answer

### Features
- impressions_90d
- clicks_90d
- avg_position_90d
- search_volume

### Label
- ctr_90d

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
Fields that directly reveal the prediction target or future information are excluded to avoid data leakage.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily_sample']}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 3. Verify it with queries (grain, counts, missing values, windows)



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {TABLES['fact_daily_sample']}
""").df()

,total_rows
0,11694072


In [14]:
con.sql(f"""
SELECT
COUNT(*) AS missing_report_dates
FROM {TABLES['fact_daily_sample']}
WHERE report_date IS NULL
""").df()

,missing_report_dates
0,0


In [11]:
con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily_sample']}
""").df()

,start_date,end_date
0,2026-06-01,2026-06-30


In [12]:
con.sql(f"""
SELECT
COUNT(DISTINCT client_hash_id) AS clients
FROM {TABLES['fact_daily_sample']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients
0,65


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
COUNT(DISTINCT client_hash_id) AS clients
FROM {TABLES['fact_daily_sample']}
""").df()

,clients
0,65


## Self-check

Before you submit, confirm each line honestly:

- [ ok] Every section above is filled — markdown thinking AND the code that backs it
- [ ok] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ok] No client names, URLs, or private queries anywhere
- [ ok] My claims use careful words: observed, measured, directional, decision-support
- [ok ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.